# Emoji-preserving sentiment ablation

Prepared 9 November 2026 (Asia/Calcutta). **Generated, unexecuted notebook.**

Test whether preserving symbol context improves sentiment on an annotated emoji-bearing movie cohort.

Inputs: actual human-labelled movie training/evaluation manifests and paired completed model reports.

There is no fixed emoji sentiment table. The same symbol may be positive, sarcastic, negative or non-evaluative depending on context. Full emoji grapheme segmentation is not claimed by the conservative detector.

No datasets, dependencies or model weights are fetched. No model is trained by running all cells with the default flag. Fill only real, approved local paths. Do not store private review text in committed outputs. Source presence is not Phase 2/3 acceptance.

Read [the Phase 3 runbook](../documentation/phase3/RUNBOOK.md) first.

In [ ]:
from pathlib import Path
import sys

# Explicit opt-in for later local inspection; this delivery has not executed cells.
RUN_APPROVED_EXPERIMENTS = False
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "ARCHITECTURE.md").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the D:/BDA_TA workspace.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workspace:", ROOT)
print("Experiment execution enabled:", RUN_APPROVED_EXPERIMENTS)

In [ ]:
CONFIG_PATH = ROOT / "ml/multilingual_sentiment/config.yaml"
if RUN_APPROVED_EXPERIMENTS:
    from ml.common.config import load_config
    from ml.common.data import load_experiment
    from ml.informal_language.preprocess import feature_view
    bundle = load_experiment(load_config(CONFIG_PATH))
    changed = 0
    for record in bundle.records["train"][:128]:
        original = record.read_text()
        preserve_view = feature_view(original, emoji_policy="preserve")
        ablated_view = feature_view(original, emoji_policy="remove_for_ablation")
        assert preserve_view.original == ablated_view.original == original
        changed += preserve_view.text != ablated_view.text
    print({"bounded_train_rows": min(128, len(bundle.records["train"])),
           "changed_by_symbol_ablation": changed})
else:
    print("No text or symbols inspected; source text remains untouched.")

## Prespecified paired comparison

Use the same task, split hashes, base model, training budget and seed policy. Compare `preprocessing.emoji_policy=preserve` with `remove_for_ablation` in separate approved runs. The implementation removes conservative Unicode symbol runs, so describe it as a symbol-removal ablation rather than a complete emoji parser.

Select hyperparameters on tuning before final test. Report overall and explicitly annotated emoji-heavy cohort support/metrics; test each frozen ablation once. Preserve alignment to original text for later aspect extraction.

In [ ]:
PRESERVE_REPORT = None
REMOVAL_REPORT = None
if RUN_APPROVED_EXPERIMENTS and PRESERVE_REPORT and REMOVAL_REPORT:
    from ml.informal_language.evaluate import compare_reports
    comparison = compare_reports([ROOT / PRESERVE_REPORT, ROOT / REMOVAL_REPORT])
    print(comparison)
else:
    print("No paired real reports supplied; no emoji-related improvement is claimed.")

## Interpretation and acceptance

Record measured results, source/model licenses, exact config and manifest hashes, code revision, dependency versions, hardware and unsupported cohorts after the approved runs. Keep the final held-out set separate from model selection. No result has been measured in this notebook. Update the model card from actual reports; missing evidence remains missing.